# 08 - Lakeflow Jobs: branching, loops, retries, and triggers

Prerequisites / execution context: Run 00 and 01. Use the bundle sources for actual job tasks. Jobs task values require a job run context. This notebook provides setup and verification instructions.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R37: Lakeflow Jobs](https://docs.databricks.com/aws/en/jobs/); [R38: If/else tasks](https://docs.databricks.com/aws/en/jobs/tasks/if-else); [R39: For each tasks](https://docs.databricks.com/aws/en/jobs/tasks/for-each); [R40: Dynamic value references](https://docs.databricks.com/aws/en/jobs/dynamic-value-references); [R41: Task values](https://docs.databricks.com/aws/en/jobs/task-values); [R42: Job triggers](https://docs.databricks.com/aws/en/jobs/triggers); [R43: Configure job tasks and timeouts](https://docs.databricks.com/aws/en/jobs/configure-task).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Build or deploy the job
Deploy bundle/databricks.yml using lab 09, or create the same tasks in the UI. The parameter join_type defaults to left. Tasks: quality -> choose_join -> left_join on true / inner_join on false -> publish when at least one branch succeeds -> per_country loop. Use the exact branch outcome dependency, not only a plain dependency on choose_join.


## 2. Three namespaces
Job parameter: {{job.parameters.join_type}}. Task-produced value: {{tasks.quality.values.valid_rows}}. Bundle deployment variable: ${var.catalog}. Dynamic references are resolved in job configuration and passed into notebook parameters; writing those braces inside ordinary Python does not resolve them. Official Databricks documentation: [R40: Dynamic value references](https://docs.databricks.com/aws/en/jobs/dynamic-value-references); [R48: Bundle variables](https://docs.databricks.com/aws/en/dev-tools/bundles/variables).


## 3. Run twice and verify
First run join_type=left: left branch executes and inner branch is excluded; output has 4 rows. Second run join_type=inner: inner branch executes and left is excluded; output has 2 rows. In the run details inspect the evaluated condition and actual task parameters. Compare STRING values exactly: left versus Left differ. The two branches write the same target safely because exactly one branch runs and max_concurrent_runs is 1.


In [ ]:
# Run after each job, from this notebook; job sources write to a separate target.
CHECK_JOB=False
if CHECK_JOB:
    out=spark.table(f"{GOLD}.job_join_result")
    out.orderBy("order_id").show()
    assert out.count() in (2,4)


## 4. Failure and repair exercise
Temporarily make the quality task assertion fail in a development copy. Observe upstream failure and blocked downstream tasks. Repair after fixing the defect and inspect which tasks rerun. Retries suit transient errors; invalid data needs a policy. Configure a 900-second task timeout and distinguish it from a warning threshold and cluster auto-termination. Official Databricks documentation: [R43: Configure job tasks and timeouts](https://docs.databricks.com/aws/en/jobs/configure-task); [R44: Monitor Jobs](https://docs.databricks.com/aws/en/jobs/monitor).


## 5. Trigger selection and extra task types
Use a schedule for a fixed reporting time, file arrival for unpredictable file delivery, and table update for changed upstream tables. Confirm path permissions and feature availability. In a development copy add a SQL task querying the Gold table (select a SQL warehouse), a pipeline task referencing lab 07, and a dashboard task referencing a saved dashboard. Set dependencies so dashboard refresh follows successful Gold publication. Record their compute and permissions requirements. Official Databricks documentation: [R42: Job triggers](https://docs.databricks.com/aws/en/jobs/triggers); [R43: Configure job tasks and timeouts](https://docs.databricks.com/aws/en/jobs/configure-task).


## 6. Loop check
The bundle For each inputs contain IT and FR; the nested notebook receives {{input.country}} and writes separate country tables. Concurrency is 1 for a small account. A For each task cannot directly nest another For each. Official Databricks documentation: [R39: For each tasks](https://docs.databricks.com/aws/en/jobs/tasks/for-each).
